In [1]:
import os
from pathlib import Path

# Get the current directory of the notebook
notebook_path = Path.cwd()

ROOT = notebook_path.parent.parent.parent

# Change the Working Directory for the whole process
os.chdir(ROOT)

print(f"Current Working Directory fixed to: {os.getcwd()}")

Current Working Directory fixed to: /srv/homes/onbo10/thesis_main


In [ ]:
from src.Geometry.triangulation.keypoints_triangulation import *
from src.Keypoints_detection.inference.inferencer import keypointsDetectionInferencer, run_multi_tool_inference
from src.Keypoints_detection.training.utils import get_device
from ultralytics import YOLO
from src.Keypoints_detection.Top_down_keypoints_detection_pipline import KeypointDetectionPipeline
from mmpose.apis import init_model
from utilities.visualizer_triangulation import TriangulationVisualizer
from src.Geometry.triangulation.triangulator import Triangulator
from src.Geometry.triangulation.triangulation_utils import *
import os
import numpy as np
import json


#### Run Vitpose pipeline on the whole test set and dave results

In [ ]:
data_root_l='data/SurgPose/SurgPose_for_HRNet/Extracted' 
data_root_r ='data/SurgPose/SurgPose_for_HRNet/Extracted_right_test' 
split_file='data/SurgPose/SurgPose_for_HRNet/Extracted/video_split.yaml'  
org_dataset_path =  'data/SurgPose/SurgPose_for_HRNet'
json_kpts_path='results/Keypoints_detection/inference_results/triangulation/vitpose_pipeline/vitpose_triangulation_testset_rectified_kpts.json'

In [4]:
test_video_list, test_paths_l, test_paths_r = get_paths_and_video_lists(data_root_l, data_root_r, split_file)

Total images in directory: 11356
Images to process (Test Split): 2004


In [5]:
device = get_device()

In [6]:
det_model = YOLO("results/Keypoints_detection/training_results/YOLO_trainings/YOLO_Object_Experiment1/weights/best.pt")
det_model.eval();

In [7]:

config_file = 'configs/Vitpose/vitpose_surg_7kpt.py'
checkpoint_file = 'results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth'
model_type_vitpose='vitpose'
pose_model_vitpose = init_model(config_file, checkpoint_file, device=device)
pose_model_vitpose.eval();

Loads checkpoint by local backend from path: results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth


In [8]:

vitpose_pipeline = KeypointDetectionPipeline(det_model,pose_model_vitpose,model_type_vitpose, device= device)


In [9]:

inferencer =  keypointsDetectionInferencer(vitpose_pipeline,'pipeline',device)


In [10]:

tri = Triangulator(num_keypoints=7)

In [11]:

# Run the pipeline
all_results_vitpose = triangulate_and_save_all(
    inferencer=inferencer,
    triangulator=tri,
    test_paths_l=test_paths_l,
    test_paths_r=test_paths_r,
    test_video_list=test_video_list,
    org_dataset_path= org_dataset_path, 
    img_size=(986,1400),
    max_tools=2,
    save_path=json_kpts_path
)



Running batch inference on Test frames...
Processing Video: 000004
Processing Video: 000030
Processing Video: 000033
Processing Video: 000017
Processing Video: 000001
Processing Video: 000007
Triangulation complete. Log saved to /srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/triangulation/vitpose_pipeline/vitpose_triangulation_testset_rectified_kpts_4.json
